In [ ]:
# Libraries for GNN recommender system
import numpy as np
import pandas as pd
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
import networkx as nx
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score, recall_score
import matplotlib.pyplot as plt
import random

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# Check versions
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Torch Geometric:', Data)
print('NetworkX:', nx.__version__)
print('Numpy:', np.__version__)
print('Pandas:', pd.__version__)
# print('Matplotlib:', plt.__version__)


Torch version: 2.10.0+cpu
CUDA available: False
Torch Geometric: <class 'torch_geometric.data.data.Data'>
NetworkX: 3.4.2
Numpy: 1.24.4
Pandas: 2.3.3


In [45]:
import torch
import torch_geometric
print("Torch version:", torch.__version__)
print("Torch Geometric version:", torch_geometric.__version__)

Torch version: 2.10.0+cpu
Torch Geometric version: 2.8.0


In [46]:

# !pip freeze > requirements.txt

import sys
sys.version


'3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]'

In [47]:
# Load raw data
books = pd.read_csv('../data/original/Books.csv')
users = pd.read_csv('../data/original/Users.csv')
ratings = pd.read_csv('../data/original/Ratings.csv')

# --- Data Cleaning ---
# Remove zero ratings
ratings = ratings[ratings['Book-Rating'] != 0]
# Drop missing values
ratings = ratings.dropna(subset=['User-ID', 'ISBN', 'Book-Rating'])
ratings = ratings.drop_duplicates(subset=['User-ID', 'ISBN'])
users = users.dropna(subset=['User-ID'])
users = users[(users['Age'].fillna(0) > 5) & (users['Age'].fillna(0) < 100)]
books = books.dropna(subset=['ISBN', 'Book-Title'])
# Merge ratings with valid users and books
ratings = ratings.merge(users[['User-ID']], on='User-ID', how='inner')
ratings = ratings.merge(books[['ISBN']], on='ISBN', how='inner')
ratings.reset_index(drop=True, inplace=True)
users.reset_index(drop=True, inplace=True)
books.reset_index(drop=True, inplace=True)

print('Cleaned Ratings:', ratings.shape)
print('Cleaned Users:', users.shape)
print('Cleaned Books:', books.shape)

C:\Users\Mukand\AppData\Local\Temp\ipykernel_22292\2113491159.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('../data/original/Books.csv')


Cleaned Ratings: (245091, 3)
Cleaned Users: (166815, 3)
Cleaned Books: (271360, 8)


In [48]:
print(ratings.head())
print(users.head())
print(books.head())

   User-ID        ISBN  Book-Rating
0   276729  052165615X            3
1   276729   521795028            6
2   276747    60517794            9
3   276747   671537458            9
4   276747   679776818            8
   User-ID                        Location   Age
0        2       stockton, california, usa  18.0
1        4       porto, v.n.gaia, portugal  17.0
2        6   santa monica, california, usa  61.0
3       10      albacete, wisconsin, spain  26.0
4       11  melbourne, victoria, australia  14.0
        ISBN                                         Book-Title  \
0  195153448                                Classical Mythology   
1    2005018                                       Clara Callan   
2   60973129                               Decision in Normandy   
3  374157065  Flu: The Story of the Great Influenza Pandemic...   
4  393045218                             The Mummies of Urumchi   

            Book-Author Year-Of-Publication                   Publisher  \
0    Mark P.

In [49]:
# Find the ISBNs in books but not in ratings
not_common_in_ratings = set(books['ISBN']).difference(set(ratings['ISBN']))

# # Print the ISBNs not common in ratings
# print("ISBNs in books but not in ratings:")
# for isbn in not_common_in_ratings:
#     print(isbn)

# Print the count of ISBNs not common in ratings
print("Number of ISBNs in books but not in ratings:", len(not_common_in_ratings))

Number of ISBNs in books but not in ratings: 158428


In [50]:
print(27774-23852)

3922


In [51]:
# Find the common User-IDs
common_users = set(ratings['User-ID']).intersection(set(users['User-ID']))

# Count the number of common User-IDs
num_common_users = len(common_users)

# Print the number of common User-IDs
print("Number of common User-IDs:", num_common_users)

Number of common User-IDs: 36496


In [52]:
# Find the User-IDs in ratings but not in users
not_common_in_users = set(ratings['User-ID']).difference(set(users['User-ID']))

# Print the count of User-IDs not common in users
print("Number of User-IDs in ratings but not in users:", len(not_common_in_users))

Number of User-IDs in ratings but not in users: 0


In [53]:
# Find the User-IDs in users but not in ratings
not_common_in_ratings = set(users['User-ID']).difference(set(ratings['User-ID']))

# Print the count of User-IDs not common in ratings
print("Number of User-IDs in users but not in ratings:", len(not_common_in_ratings))

Number of User-IDs in users but not in ratings: 130319


In [54]:
print('unique isbns', ratings['ISBN'].nunique())

unique isbns 112932


In [55]:
print('unique isbns', books['ISBN'].nunique())

unique isbns 271360


In [57]:
print(ratings['Book-Rating'].value_counts())

Book-Rating
8     59045
10    49102
9     41414
7     41341
5     25092
6     19394
4      4529
3      3002
2      1389
1       783
Name: count, dtype: int64


### removing unnecessary columns

In [58]:
# columns_to_remove = ['Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'Location', 'Age', 'Year-Of-Publication', 'Publisher']

# books_cleaned = books.drop(columns=columns_to_remove)

display(ratings.head())

display(ratings.shape)

,User-ID,ISBN,Book-Rating
0,276729,052165615X,3
1,276729,521795028,6
2,276747,60517794,9
3,276747,671537458,9
4,276747,679776818,8


(245091, 3)

In [59]:

unique_user_ids = ratings['User-ID'].nunique()
print("Unique User-ID:", unique_user_ids)

# Check unique ISBN
unique_isbns = ratings['ISBN'].nunique()
print("Unique ISBN:", unique_isbns)

# # Check unique Book_title
# unique_title = ratings['Book-Title'].nunique()
# print("Unique ISBN:", unique_title)

Unique User-ID: 36496
Unique ISBN: 112932


In [60]:
ratings.describe()

,User-ID,Book-Rating
count,245091.000000,245091.000000
mean,126506.225524,7.744062
std,72090.902930,1.809389
min,19.000000,1.000000
25%,66572.000000,7.000000
50%,123649.000000,8.000000
75%,187517.000000,9.000000
max,278852.000000,10.000000


In [61]:
# Range of User-ID
user_id_min = ratings['User-ID'].min()
user_id_max = ratings['User-ID'].max()
print("Range of User-ID: {} - {}".format(user_id_min, user_id_max))

# Range of ISBN
isbn_min = ratings['ISBN'].min()
isbn_max = ratings['ISBN'].max()
print("Range of ISBN: {} - {}".format(isbn_min, isbn_max))

Range of User-ID: 19 - 278852
Range of ISBN: 000104799X - B000234N3A


In [62]:
# ratings = ratings.merge(books, on='ISBN')

# books_cleaned['Book-Title'] = books_cleaned['Book-Title'].replace(to_replace='\$', value='', regex=True)

user_node =ratings['User-ID'].unique()
book_node =ratings['ISBN'].unique()

display(book_node)
display(user_node)

array(['052165615X', '521795028', '60517794', ..., '970349505',
       '1552790207', '189395630X'], dtype=object)

array([276729, 276747, 276748, ..., 250738, 250750, 250751], dtype=int64)

In [63]:
ratings = ratings[["User-ID","Book-Rating","ISBN"]]

### 2. Index Mapping
Map `User-ID` and `ISBN` to integer indices for graph construction:


In [64]:
# Index Mapping
user_to_index = {user: idx for idx, user in enumerate(ratings['User-ID'].unique())}
book_to_index = {book: idx for idx, book in enumerate(ratings['ISBN'].unique())}
ratings['user_index'] = ratings['User-ID'].map(user_to_index)
ratings['book_index'] = ratings['ISBN'].map(book_to_index)

print(f"Total Users: {len(user_to_index)}, Total Books: {len(book_to_index)}")


Total Users: 36496, Total Books: 112932


### 3. Graph Construction
Build edge list and node features:


In [65]:
import torch

edge_index = torch.tensor([ratings['user_index'].values, ratings['book_index'].values], dtype=torch.long)
edge_attr = torch.tensor(ratings['Book-Rating'].values, dtype=torch.float)

num_users = len(user_to_index)
num_books = len(book_to_index)
features = torch.arange(num_users + num_books, dtype=torch.long)

# Note: edge_index needs an offset for the books since features are concatenated 
# User nodes are 0 to num_users-1, Book nodes are num_users to num_users+num_books-1
offset_edge_index = edge_index.clone()
offset_edge_index[1, :] += num_users


C:\Users\Mukand\AppData\Local\Temp\ipykernel_22292\2019041634.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([ratings['user_index'].values, ratings['book_index'].values], dtype=torch.long)


### 4. PyTorch Geometric Data Object


In [66]:
from torch_geometric.data import Data

data = Data(x=features, edge_index=offset_edge_index, edge_attr=edge_attr, num_nodes=num_users+num_books)
print(data)


Data(x=[149428], edge_index=[2, 245091], edge_attr=[245091], num_nodes=149428)


### 5. Train/Test Split


In [67]:
import numpy as np
from sklearn.model_selection import train_test_split

train_mask, test_mask = train_test_split(np.arange(data.num_edges), test_size=0.2, random_state=42)
train_mask = torch.tensor(train_mask)
test_mask = torch.tensor(test_mask)

print(f"Train edges: {len(train_mask)}, Test edges: {len(test_mask)}")


Train edges: 196072, Test edges: 49019


### 6. Model Definition
Using GCNConv layers:


In [68]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GNNRecSys(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64):
        super(GNNRecSys, self).__init__()
        # Initial Embeddings
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
        
        # GCN layers
        self.conv1 = GCNConv(embedding_dim, 64)
        self.conv2 = GCNConv(64, 32)
        
        # Output layers (Predicting rating)
        self.linear1 = nn.Linear(32 * 2, 64)
        self.linear2 = nn.Linear(64, 1)

    def forward(self, edge_index, user_indices, book_indices):
        # We start directly from embedding layer instead of `x` (one-hot matrices are very sparse and memory intensive)
        x = self.embedding.weight 
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        # Extract the node representations and concatenate them to predict edge attributes (ratings)
        user_h = x[user_indices]
        book_h = x[book_indices]
        
        out = torch.cat([user_h, book_h], dim=1)
        out = F.relu(self.linear1(out))
        out = F.dropout(out, p=0.2, training=self.training)
        out = self.linear2(out)
        
        return out.squeeze()


### 7. Training and Evaluation


In [70]:
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_NODES = num_users + num_books
model = GNNRecSys(NUM_NODES, embedding_dim=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

data = data.to(device)
train_edge_index = data.edge_index[:, train_mask]
train_edge_attr = data.edge_attr[train_mask]
test_edge_index = data.edge_index[:, test_mask]
test_edge_attr = data.edge_attr[test_mask]

epochs = 100
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass on train edges
    user_indices = train_edge_index[0]
    book_indices = train_edge_index[1]
    
    preds = model(data.edge_index, user_indices, book_indices)
    loss = criterion(preds, train_edge_attr)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            t_user_indices = test_edge_index[0]
            t_book_indices = test_edge_index[1]
            t_preds = model(data.edge_index, t_user_indices, t_book_indices)
            
            val_loss = criterion(t_preds, test_edge_attr)
            val_rmse = torch.sqrt(val_loss)
            
        print(f"Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Val RMSE: {val_rmse.item():.4f}")



Epoch 010 | Train Loss: 9.3282 | Val RMSE: 2.7665
Epoch 020 | Train Loss: 5.0879 | Val RMSE: 2.2809
Epoch 030 | Train Loss: 3.8415 | Val RMSE: 1.8413
Epoch 040 | Train Loss: 3.3466 | Val RMSE: 1.8636
Epoch 050 | Train Loss: 3.0149 | Val RMSE: 1.7491
Epoch 060 | Train Loss: 2.8026 | Val RMSE: 1.7545
Epoch 070 | Train Loss: 2.6742 | Val RMSE: 1.7800
Epoch 080 | Train Loss: 2.5317 | Val RMSE: 1.7671
Epoch 090 | Train Loss: 2.5303 | Val RMSE: 1.7561
Epoch 100 | Train Loss: 2.3469 | Val RMSE: 1.7570


### 8. Recommendation Function
Generate Top-N Recommendations


In [71]:
from tabulate import tabulate

def recommend_books(user_id, top_n=5):
    model.eval()
    
    if user_id not in user_to_index:
        return []
    
    u_idx = user_to_index[user_id]
    
    # Get all books the user has NOT rated yet
    user_rated_books = set(ratings[ratings['User-ID'] == user_id]['ISBN'].values)
    all_books = ratings['ISBN'].unique()
    unrated_books = [b for b in all_books if b not in user_rated_books]
    
    # Only keep unrated books that are in the book_to_index mapping
    unrated_books_indices = [book_to_index[b] + num_users for b in unrated_books if b in book_to_index]
    
    # Prepare batch
    u_tensor = torch.tensor([u_idx] * len(unrated_books_indices), dtype=torch.long).to(device)
    b_tensor = torch.tensor(unrated_books_indices, dtype=torch.long).to(device)
    
    with torch.no_grad():
        preds = model(data.edge_index, u_tensor, b_tensor)
        
    # Get top N max predictions
    top_scores, top_indices = torch.topk(preds, top_n)
    
    recommended_isbns = [unrated_books[i.item()] for i in top_indices]
    
    # Track scores for requested format
    rec_results = []
    
    for i, isbn in enumerate(recommended_isbns):
        # Retrieve Titles
        title_matches = books[books['ISBN'] == isbn]['Book-Title'].values
        if len(title_matches) > 0:
            rec_results.append([title_matches[0], float(top_scores[i].item())])
            
    # Format the table
    print("Here is the data with the index column removed:")
    print("")
    print(tabulate(rec_results, headers=['Recommended Book', 'Similarity Score'], tablefmt='grid'))
    print("")
    
    return rec_results

# Example Usage
sample_users = [ratings['User-ID'].iloc[0], ratings['User-ID'].iloc[10], ratings['User-ID'].iloc[20]]
for i, u in enumerate(sample_users):
    recommend_books(u, top_n=5)


Here is the data with the index column removed:

+--------------------------------------------+--------------------+
| Recommended Book                           |   Similarity Score |
+============================================+====================+
| Anna Hanna Och Johanna                     |            8.78155 |
+--------------------------------------------+--------------------+
| Luces De Bohemia (Nueva Austral, No 1)     |            8.15358 |
+--------------------------------------------+--------------------+
| The Alleys of Eden: A Novel                |            7.98119 |
+--------------------------------------------+--------------------+
| Stick A Geranium In Your Hat And Be Happy! |            7.9798  |
+--------------------------------------------+--------------------+
| Sellevision: A Novel                       |            7.7593  |
+--------------------------------------------+--------------------+

Here is the data with the index column removed:

+----------------